In [ ]:
import torch
import os

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Torch version:", torch.__version__)
else:
    print("GPU yok.")

CUDA available: True
GPU: Tesla T4
Torch version: 2.10.0+cu128


In [2]:
import pandas as pd
import os

test_path = "/data/processed/strategy_b/test.csv"

print("Test file exists:", os.path.exists(test_path))

test_df = pd.read_csv(test_path)

print("Shape:", test_df.shape)
print("Columns:", test_df.columns.tolist())
print("\nLabel distribution:")
print(test_df["label"].value_counts().sort_index())

Test file exists: True
Shape: (1447, 8)
Columns: ['input_text', 'label', 'Score', 'soru', 'context', 'cevap', 'kaynak', 'veri türü']

Label distribution:
label
0    885
1    562
Name: count, dtype: int64


In [3]:
from transformers import AutoProcessor

model_id = "google/gemma-4-E2B-it"

processor = AutoProcessor.from_pretrained(model_id)

print("Processor loaded:", model_id)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

You are using a model of type gemma4 to instantiate a model of type . This is not supported for all configurations of models and can yield errors.


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

Processor loaded: google/gemma-4-E2B-it


In [4]:
from transformers import AutoModelForCausalLM
import torch

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float16,
    device_map="auto"
)

model.eval()

print("Model loaded:", model_id)

ValueError: The checkpoint you are trying to load has model type `gemma4` but Transformers does not recognize this architecture. This could be because of an issue with the checkpoint, or because your version of Transformers is out of date.

You can update Transformers with the command `pip install --upgrade transformers`. If this does not work, and the checkpoint is very new, then there may not be a release version that supports this model yet. In this case, you can get the most up-to-date code by installing Transformers from source with the command `pip install git+https://github.com/huggingface/transformers.git`

In [5]:
!pip install -U git+https://github.com/huggingface/transformers.git -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [1]:
import torch
import transformers

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Torch version:", torch.__version__)
print("Transformers version:", transformers.__version__)

CUDA available: True
GPU: Tesla T4
Torch version: 2.10.0+cu128
Transformers version: 5.8.0.dev0


In [2]:
import pandas as pd
import os

test_path = "/data/processed/strategy_b/test.csv"

print("Test file exists:", os.path.exists(test_path))

test_df = pd.read_csv(test_path)

print("Shape:", test_df.shape)
print("Columns:", test_df.columns.tolist())
print("\nLabel distribution:")
print(test_df["label"].value_counts().sort_index())

Test file exists: True
Shape: (1447, 8)
Columns: ['input_text', 'label', 'Score', 'soru', 'context', 'cevap', 'kaynak', 'veri türü']

Label distribution:
label
0    885
1    562
Name: count, dtype: int64


In [3]:
from transformers import AutoProcessor, AutoModelForCausalLM
import torch

model_id = "google/gemma-4-E2B-it"

processor = AutoProcessor.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float16,
    device_map="auto"
)

model.eval()

print("Model loaded:", model_id)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

Model loaded: google/gemma-4-E2B-it


In [4]:
sample_label_0 = test_df[test_df["label"] == 0].sample(n=10, random_state=42)
sample_label_1 = test_df[test_df["label"] == 1].sample(n=10, random_state=42)

zero_shot_sample = pd.concat([sample_label_0, sample_label_1])
zero_shot_sample = zero_shot_sample.sample(frac=1, random_state=42).reset_index(drop=True)

print("Sample shape:", zero_shot_sample.shape)
print("Sample label distribution:")
print(zero_shot_sample["label"].value_counts().sort_index())

Sample shape: (20, 8)
Sample label distribution:
label
0    10
1    10
Name: count, dtype: int64


In [5]:
def create_zero_shot_prompt_v3(input_text):
    prompt = f"""
You are grading a Turkish open-ended student answer.

Use the following binary grading rule:

0 = not high-quality.
Choose 0 if the answer is only partially correct, too general, weakly explained, incomplete, contains unsupported claims, or does not fully use the given context.

1 = high-quality.
Choose 1 only if the answer is clearly correct, complete, well explained, directly relevant to the question, and well supported by the context.

Important rules:
- Do not give 1 just because the answer is fluent or long.
- If you are unsure, choose 0.
- Be strict. Label 1 should be reserved for very strong answers.

Return only one number: 0 or 1.

Input:
{input_text}

Label:
"""
    return prompt.strip()


def parse_prediction(response):
    response = str(response).strip()

    if response.startswith("0"):
        return 0
    elif response.startswith("1"):
        return 1
    elif "0" in response and "1" not in response:
        return 0
    elif "1" in response and "0" not in response:
        return 1
    else:
        return None

In [6]:
def generate_gemma_prediction(prompt, max_new_tokens=5):
    messages = [
        {"role": "user", "content": prompt}
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = processor(
        text=text,
        return_tensors="pt"
    ).to(model.device)

    input_len = inputs["input_ids"].shape[-1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=processor.tokenizer.eos_token_id
        )

    generated_ids = outputs[0][input_len:]
    response = processor.decode(generated_ids, skip_special_tokens=True)

    return response.strip()

In [7]:
example = zero_shot_sample.iloc[0]
example_prompt = create_zero_shot_prompt_v3(example["input_text"])

response = generate_gemma_prediction(example_prompt)
parsed = parse_prediction(response)

print("Raw response:", repr(response))
print("Parsed prediction:", parsed)
print("True label:", example["label"])

Raw response: '0'
Parsed prediction: 0
True label: 0


In [8]:
predictions_20_gemma_v3 = []
raw_responses_20_gemma_v3 = []

for idx, row in zero_shot_sample.iterrows():
    prompt = create_zero_shot_prompt_v3(row["input_text"])
    response = generate_gemma_prediction(prompt)
    pred = parse_prediction(response)

    raw_responses_20_gemma_v3.append(response)
    predictions_20_gemma_v3.append(pred)

    print(f"Example {idx+1}/20")
    print("True label:", row["label"])
    print("Raw response:", repr(response))
    print("Parsed prediction:", pred)
    print("-" * 40)

zero_shot_sample_results_gemma_v3 = zero_shot_sample.copy()
zero_shot_sample_results_gemma_v3["raw_response"] = raw_responses_20_gemma_v3
zero_shot_sample_results_gemma_v3["prediction"] = predictions_20_gemma_v3

Example 1/20
True label: 0
Raw response: '0'
Parsed prediction: 0
----------------------------------------
Example 2/20
True label: 1
Raw response: '0'
Parsed prediction: 0
----------------------------------------
Example 3/20
True label: 1
Raw response: '1'
Parsed prediction: 1
----------------------------------------
Example 4/20
True label: 0
Raw response: '1'
Parsed prediction: 1
----------------------------------------
Example 5/20
True label: 0
Raw response: '0'
Parsed prediction: 0
----------------------------------------
Example 6/20
True label: 0
Raw response: '1'
Parsed prediction: 1
----------------------------------------
Example 7/20
True label: 1
Raw response: '0'
Parsed prediction: 0
----------------------------------------
Example 8/20
True label: 0
Raw response: '1'
Parsed prediction: 1
----------------------------------------
Example 9/20
True label: 1
Raw response: '1'
Parsed prediction: 1
----------------------------------------
Example 10/20
True label: 1
Raw respo

In [9]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

y_true_20_gemma_v3 = zero_shot_sample_results_gemma_v3["label"]
y_pred_20_gemma_v3 = zero_shot_sample_results_gemma_v3["prediction"]

accuracy_20_gemma_v3 = accuracy_score(y_true_20_gemma_v3, y_pred_20_gemma_v3)
macro_f1_20_gemma_v3 = f1_score(y_true_20_gemma_v3, y_pred_20_gemma_v3, average="macro", zero_division=0)
weighted_f1_20_gemma_v3 = f1_score(y_true_20_gemma_v3, y_pred_20_gemma_v3, average="weighted", zero_division=0)

print("Gemma 4 E2B-it Zero-Shot Results - 20 Sample - Prompt v3")
print("Accuracy:", accuracy_20_gemma_v3)
print("Macro F1:", macro_f1_20_gemma_v3)
print("Weighted F1:", weighted_f1_20_gemma_v3)

print("\nConfusion Matrix:")
print(confusion_matrix(y_true_20_gemma_v3, y_pred_20_gemma_v3))

print("\nClassification Report:")
print(classification_report(y_true_20_gemma_v3, y_pred_20_gemma_v3, zero_division=0))

Gemma 4 E2B-it Zero-Shot Results - 20 Sample - Prompt v3
Accuracy: 0.55
Macro F1: 0.5488721804511278
Weighted F1: 0.5488721804511277

Confusion Matrix:
[[5 5]
 [4 6]]

Classification Report:
              precision    recall  f1-score   support

           0       0.56      0.50      0.53        10
           1       0.55      0.60      0.57        10

    accuracy                           0.55        20
   macro avg       0.55      0.55      0.55        20
weighted avg       0.55      0.55      0.55        20



In [10]:
sample_100_label_0 = test_df[test_df["label"] == 0].sample(n=50, random_state=42)
sample_100_label_1 = test_df[test_df["label"] == 1].sample(n=50, random_state=42)

zero_shot_sample_100 = pd.concat([sample_100_label_0, sample_100_label_1])
zero_shot_sample_100 = zero_shot_sample_100.sample(frac=1, random_state=42).reset_index(drop=True)

print("Sample shape:", zero_shot_sample_100.shape)
print("Sample label distribution:")
print(zero_shot_sample_100["label"].value_counts().sort_index())

Sample shape: (100, 8)
Sample label distribution:
label
0    50
1    50
Name: count, dtype: int64


In [11]:
predictions_100_gemma_v3 = []
raw_responses_100_gemma_v3 = []

for idx, row in zero_shot_sample_100.iterrows():
    prompt = create_zero_shot_prompt_v3(row["input_text"])
    response = generate_gemma_prediction(prompt)
    pred = parse_prediction(response)

    raw_responses_100_gemma_v3.append(response)
    predictions_100_gemma_v3.append(pred)

    if (idx + 1) % 10 == 0:
        print(f"Processed {idx + 1}/100")

zero_shot_sample_100_results_gemma_v3 = zero_shot_sample_100.copy()
zero_shot_sample_100_results_gemma_v3["raw_response"] = raw_responses_100_gemma_v3
zero_shot_sample_100_results_gemma_v3["prediction"] = predictions_100_gemma_v3

print("Done.")

Processed 10/100
Processed 20/100
Processed 30/100
Processed 40/100
Processed 50/100
Processed 60/100
Processed 70/100
Processed 80/100
Processed 90/100
Processed 100/100
Done.


In [12]:
y_true_100_gemma_v3 = zero_shot_sample_100_results_gemma_v3["label"]
y_pred_100_gemma_v3 = zero_shot_sample_100_results_gemma_v3["prediction"]

accuracy_100_gemma_v3 = accuracy_score(y_true_100_gemma_v3, y_pred_100_gemma_v3)
macro_f1_100_gemma_v3 = f1_score(y_true_100_gemma_v3, y_pred_100_gemma_v3, average="macro", zero_division=0)
weighted_f1_100_gemma_v3 = f1_score(y_true_100_gemma_v3, y_pred_100_gemma_v3, average="weighted", zero_division=0)

print("Gemma 4 E2B-it Zero-Shot Results - 100 Sample - Prompt v3")
print("Accuracy:", accuracy_100_gemma_v3)
print("Macro F1:", macro_f1_100_gemma_v3)
print("Weighted F1:", weighted_f1_100_gemma_v3)

print("\nConfusion Matrix:")
print(confusion_matrix(y_true_100_gemma_v3, y_pred_100_gemma_v3))

print("\nClassification Report:")
print(classification_report(y_true_100_gemma_v3, y_pred_100_gemma_v3, zero_division=0))

/usr/local/lib/python3.12/dist-packages/sklearn/utils/_array_api.py:399: RuntimeWarning: invalid value encountered in cast
  return x.astype(dtype, copy=copy, casting=casting)


ValueError: Input y_pred contains NaN.

In [13]:
print("Prediction value counts:")
print(zero_shot_sample_100_results_gemma_v3["prediction"].value_counts(dropna=False))

failed_gemma_outputs = zero_shot_sample_100_results_gemma_v3[
    zero_shot_sample_100_results_gemma_v3["prediction"].isna()
]

print("\nParse edilemeyen örnek sayısı:", len(failed_gemma_outputs))

failed_gemma_outputs[[
    "label",
    "Score",
    "soru",
    "raw_response",
    "prediction"
]].head(20)

Prediction value counts:
prediction
1.0    72
0.0    27
NaN     1
Name: count, dtype: int64

Parse edilemeyen örnek sayısı: 1


,label,Score,soru,raw_response,prediction
6,0,8,"Yaşantı temelli öğrenme yaklaşımı, öğrenmeyi n...",C,NaN


In [14]:
gemma_100_valid = zero_shot_sample_100_results_gemma_v3.dropna(subset=["prediction"]).copy()

gemma_100_valid["prediction"] = gemma_100_valid["prediction"].astype(int)

print("Total examples:", len(zero_shot_sample_100_results_gemma_v3))
print("Valid predictions:", len(gemma_100_valid))
print("Invalid predictions:", len(zero_shot_sample_100_results_gemma_v3) - len(gemma_100_valid))

y_true_100_gemma_v3 = gemma_100_valid["label"]
y_pred_100_gemma_v3 = gemma_100_valid["prediction"]

accuracy_100_gemma_v3 = accuracy_score(y_true_100_gemma_v3, y_pred_100_gemma_v3)
macro_f1_100_gemma_v3 = f1_score(y_true_100_gemma_v3, y_pred_100_gemma_v3, average="macro", zero_division=0)
weighted_f1_100_gemma_v3 = f1_score(y_true_100_gemma_v3, y_pred_100_gemma_v3, average="weighted", zero_division=0)

print("\nGemma 4 E2B-it Zero-Shot Results - 100 Sample - Prompt v3")
print("Accuracy:", accuracy_100_gemma_v3)
print("Macro F1:", macro_f1_100_gemma_v3)
print("Weighted F1:", weighted_f1_100_gemma_v3)

print("\nConfusion Matrix:")
print(confusion_matrix(y_true_100_gemma_v3, y_pred_100_gemma_v3))

print("\nClassification Report:")
print(classification_report(y_true_100_gemma_v3, y_pred_100_gemma_v3, zero_division=0))

Total examples: 100
Valid predictions: 99
Invalid predictions: 1

Gemma 4 E2B-it Zero-Shot Results - 100 Sample - Prompt v3
Accuracy: 0.5353535353535354
Macro F1: 0.5088438308886971
Weighted F1: 0.5099964267349945

Confusion Matrix:
[[15 34]
 [12 38]]

Classification Report:
              precision    recall  f1-score   support

           0       0.56      0.31      0.39        49
           1       0.53      0.76      0.62        50

    accuracy                           0.54        99
   macro avg       0.54      0.53      0.51        99
weighted avg       0.54      0.54      0.51        99



In [15]:
gemma_result = pd.DataFrame([
    {
        "model": "Gemma-4-E2B-it",
        "evaluation_type": "zero_shot_sample",
        "sample_size": 20,
        "prompt_version": "v3",
        "accuracy": accuracy_20_gemma_v3,
        "macro_f1": macro_f1_20_gemma_v3,
        "weighted_f1": weighted_f1_20_gemma_v3,
        "valid_predictions": 20,
        "invalid_predictions": 0,
        "notes": "Model returned parseable 0/1 labels for all 20 examples."
    },
    {
        "model": "Gemma-4-E2B-it",
        "evaluation_type": "zero_shot_sample",
        "sample_size": 100,
        "prompt_version": "v3",
        "accuracy": accuracy_100_gemma_v3,
        "macro_f1": macro_f1_100_gemma_v3,
        "weighted_f1": weighted_f1_100_gemma_v3,
        "valid_predictions": len(gemma_100_valid),
        "invalid_predictions": len(zero_shot_sample_100_results_gemma_v3) - len(gemma_100_valid),
        "notes": "One output was not parseable and was excluded from metric calculation."
    }
])

gemma_result

,model,evaluation_type,sample_size,prompt_version,accuracy,macro_f1,weighted_f1,valid_predictions,invalid_predictions,notes
0,Gemma-4-E2B-it,zero_shot_sample,20,v3,0.550000,0.548872,0.548872,20,0,Model returned parseable 0/1 labels for all 20...
1,Gemma-4-E2B-it,zero_shot_sample,100,v3,0.535354,0.508844,0.509996,99,1,One output was not parseable and was excluded ...


In [17]:
gemma_zero_shot_100_predictions = zero_shot_sample_100_results_gemma_v3[[
    "input_text",
    "label",
    "Score",
    "soru",
    "cevap",
    "raw_response",
    "prediction"
]].copy()

gemma_zero_shot_100_predictions["model"] = "Gemma-4-E2B-it"
gemma_zero_shot_100_predictions["prompt_version"] = "v3"

gemma_zero_shot_100_predictions.to_csv(
    "/outputs/tables/gemma_zero_shot_100_sample_predictions_v3.csv",
    index=False
)

print("Saved:")
print("/outputs/tables/gemma_zero_shot_100_sample_predictions_v3.csv")
print("Shape:", gemma_zero_shot_100_predictions.shape)

Saved:
/outputs/tables/gemma_zero_shot_100_sample_predictions_v3.csv
Shape: (100, 9)


In [18]:
print(os.path.exists("/outputs/tables/gemma_zero_shot_100_sample_predictions_v3.csv"))

True


In [19]:
# Existing zero-shot comparison file from previous notebook/outputs may not exist in this Gemma notebook runtime.
# So we create a compact Gemma-only result file first.

gemma_result.to_csv(
    "/outputs/tables/gemma_zero_shot_results.csv",
    index=False
)

print("Saved:")
print("/outputs/tables/gemma_zero_shot_results.csv")
print("Rows:", len(gemma_result))

Saved:
/outputs/tables/gemma_zero_shot_results.csv
Rows: 2


In [20]:
print(os.path.exists("/outputs/tables/gemma_zero_shot_results.csv"))

True
